# Escaneo con Grype

## ¿Qué es Grype?

Grype es un escáner de vulnerabilidades para imágenes de contenedores y sistemas de archivos.
Analiza las dependencias del proyecto y las compara con bases de datos de vulnerabilidades conocidas.

### Bases de datos que consulta:
- NVD (National Vulnerability Database)
- GitHub Advisory Database
- OSV (Open Source Vulnerabilities)

### Severidades:
- 🔴 **Critical**: Riesgo máximo
- 🟠 **High**: Riesgo alto
- 🟡 **Medium**: Riesgo medio
- 🟢 **Low**: Riesgo bajo
- ⚪ **Negligible**: Riesgo insignificante

In [ ]:
import json
import pandas as pd
from pathlib import Path

# Cargar resultados de Grype
results_dir = Path("../data/results")
grype_files = list(results_dir.glob("*-grype.json"))

all_vulns = []

for f in grype_files:
    with open(f) as file:
        data = json.load(file)
        matches = data.get("matches", [])
        for match in matches:
            vuln = match.get("vulnerability", {})
            all_vulns.append({
                "repo": f.stem.replace("-grype", ""),
                "id": vuln.get("id", "N/A"),
                "severity": vuln.get("severity", "Unknown"),
                "package": match.get("artifact", {}).get("name", "N/A"),
                "version": match.get("artifact", {}).get("version", "N/A"),
            })

if all_vulns:
    df = pd.DataFrame(all_vulns)
    print(f"Total de vulnerabilidades: {len(df)}")
    print(f"\nDistribución por severidad:")
    print(df["severity"].value_counts())
else:
    print("No se encontraron resultados de Grype. Ejecuta el análisis primero.")